In [9]:
from google.colab import files
uploaded = files.upload()

Saving Basics of BERT and XLM-RoBERTa - PyTorch - 2.zip to Basics of BERT and XLM-RoBERTa - PyTorch - 2.zip


In [10]:
import zipfile
import io
import pandas as pd

# Unzip the uploaded file
for fn in uploaded.keys():
    zip_file = zipfile.ZipFile(io.BytesIO(uploaded[fn]), 'r')
    zip_file.extractall('/content/')
    print(f"Extracted {fn} to /content/")

Extracted Basics of BERT and XLM-RoBERTa - PyTorch - 2.zip to /content/


### Loading and Exploring the Dataset
Now, let's load the `train.csv` and `test.csv` files into pandas DataFrames and inspect their initial structure.

In [11]:
import os
import zipfile

# Define the extracted folder path
extracted_folder = '/content/Basics of BERT and XLM-RoBERTa - PyTorch/'

# List contents of the extracted directory to find the correct path
print(f"Contents of {extracted_folder}:")
for root, dirs, files in os.walk(extracted_folder):
    for name in files:
        print(os.path.join(root, name))

# Check if train.csv.zip or test.csv.zip exist and unzip them
for file_name in os.listdir(extracted_folder):
    if file_name.endswith('.zip'):
        zip_path = os.path.join(extracted_folder, file_name)
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            # Extract to the same directory or a subfolder if needed
            zip_ref.extractall(extracted_folder)
        print(f"Extracted {file_name} to {extracted_folder}")

# List contents again to confirm unzipped files
print(f"\nContents of {extracted_folder} after further unzipping:")
for root, dirs, files in os.walk(extracted_folder):
    for name in files:
        print(os.path.join(root, name))

# Now, load the training and testing data from CSV files
# The files should now be directly accessible in the extracted_folder
train_df = pd.read_csv(os.path.join(extracted_folder, 'train.csv'))
test_df = pd.read_csv(os.path.join(extracted_folder, 'test.csv'))

# Display the first few rows of the training DataFrame
print("\nTrain DataFrame Head:")
display(train_df.head())

# Display the shape of the training DataFrame
print(f"Train DataFrame Shape: {train_df.shape}")

# Display the first few rows of the test DataFrame
print("\nTest DataFrame Head:")
display(test_df.head())

# Display the shape of the test DataFrame
print(f"Test DataFrame Shape: {test_df.shape}")

Contents of /content/Basics of BERT and XLM-RoBERTa - PyTorch/:
/content/Basics of BERT and XLM-RoBERTa - PyTorch/sample_submission.csv
/content/Basics of BERT and XLM-RoBERTa - PyTorch/train.csv.zip
/content/Basics of BERT and XLM-RoBERTa - PyTorch/test.csv.zip
Extracted train.csv.zip to /content/Basics of BERT and XLM-RoBERTa - PyTorch/
Extracted test.csv.zip to /content/Basics of BERT and XLM-RoBERTa - PyTorch/

Contents of /content/Basics of BERT and XLM-RoBERTa - PyTorch/ after further unzipping:
/content/Basics of BERT and XLM-RoBERTa - PyTorch/sample_submission.csv
/content/Basics of BERT and XLM-RoBERTa - PyTorch/train.csv.zip
/content/Basics of BERT and XLM-RoBERTa - PyTorch/train.csv
/content/Basics of BERT and XLM-RoBERTa - PyTorch/test.csv
/content/Basics of BERT and XLM-RoBERTa - PyTorch/test.csv.zip

Train DataFrame Head:


,id,premise,hypothesis,lang_abv,language,label
0,5130fd2cb5,and these comments were considered in formulat...,The rules developed in the interim were put to...,en,English,0
1,5b72532a0b,These are issues that we wrestle with in pract...,Practice groups are not permitted to work on t...,en,English,2
2,3931fbe82a,Des petites choses comme celles-là font une di...,J'essayais d'accomplir quelque chose.,fr,French,0
3,5622f0c60b,you know they can't really defend themselves l...,They can't defend themselves because of their ...,en,English,0
4,86aaa48b45,ในการเล่นบทบาทสมมุติก็เช่นกัน โอกาสที่จะได้แสด...,เด็กสามารถเห็นได้ว่าชาติพันธุ์แตกต่างกันอย่างไร,th,Thai,1


Train DataFrame Shape: (12120, 6)

Test DataFrame Head:


,id,premise,hypothesis,lang_abv,language
0,c6d58c3f69,بکس، کیسی، راہیل، یسعیاہ، کیلی، کیلی، اور کولم...,"کیسی کے لئے کوئی یادگار نہیں ہوگا, کولمین ہائی...",ur,Urdu
1,cefcc82292,هذا هو ما تم نصحنا به.,عندما يتم إخبارهم بما يجب عليهم فعله ، فشلت ال...,ar,Arabic
2,e98005252c,et cela est en grande partie dû au fait que le...,Les mères se droguent.,fr,French
3,58518c10ba,与城市及其他公民及社区组织代表就IMA的艺术发展进行对话&amp,IMA与其他组织合作，因为它们都依靠共享资金。,zh,Chinese
4,c32b0d16df,Она все еще была там.,"Мы думали, что она ушла, однако, она осталась.",ru,Russian


Test DataFrame Shape: (5195, 5)


### Creating Cross-Validation Folds
To ensure robust model evaluation, we'll implement k-fold cross-validation using `StratifiedKFold`. This will split our training data into 5 folds, maintaining the proportion of samples for each class in each fold.

In [12]:
from sklearn.model_selection import StratifiedKFold
import numpy as np

# Define the number of folds
N_FOLDS = 5

# Initialize StratifiedKFold
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

# Create lists to store train and validation indices for each fold
train_indices = []
val_indices = []

# Assuming 'label' is the target column in train_df
# You may need to adjust 'train_df['label']' if your target column has a different name
for fold, (t_idx, v_idx) in enumerate(skf.split(train_df, train_df['label'])):
    train_indices.append(t_idx)
    val_indices.append(v_idx)
    print(f"Fold {fold + 1}: Train indices count: {len(t_idx)}, Validation indices count: {len(v_idx)}")

print(f"\nSuccessfully created {N_FOLDS} stratified folds for cross-validation.")

# Example: Display the class distribution for the first fold's validation set
print(f"\nClass distribution in the first fold's validation set (Fold 1):")
first_fold_val_labels = train_df.loc[val_indices[0], 'label']
display(first_fold_val_labels.value_counts(normalize=True))

Fold 1: Train indices count: 9696, Validation indices count: 2424
Fold 2: Train indices count: 9696, Validation indices count: 2424
Fold 3: Train indices count: 9696, Validation indices count: 2424
Fold 4: Train indices count: 9696, Validation indices count: 2424
Fold 5: Train indices count: 9696, Validation indices count: 2424

Successfully created 5 stratified folds for cross-validation.

Class distribution in the first fold's validation set (Fold 1):


,proportion
label,
0,0.344884
2,0.334983
1,0.320132


### 1. Understanding BERT and XLM-RoBERTa & 2. Tokenizing Text
Now, let's load the necessary tokenizers for BERT and XLM-RoBERTa and understand how to tokenize text. We'll use a sample sentence to demonstrate the tokenization process.

In [16]:
from transformers import BertTokenizer, XLMRobertaTokenizer

# Load pre-trained tokenizers
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
xlm_roberta_tokenizer = XLMRobertaTokenizer.from_pretrained('xlm-roberta-base')

print("BERT Tokenizer loaded: bert-base-uncased")
print("XLM-RoBERTa Tokenizer loaded: xlm-roberta-base")

# Example sentence for tokenization
sample_sentence = "Hello, how are you today? This is an example sentence."

print(f"\nSample Sentence: {sample_sentence}")

# Tokenize with BERT tokenizer
bert_tokens = bert_tokenizer.tokenize(sample_sentence)
print(f"\nBERT Tokens: {bert_tokens}")

# Tokenize with XLM-RoBERTa tokenizer
xlm_roberta_tokens = xlm_roberta_tokenizer.tokenize(sample_sentence)
print(f"XLM-RoBERTa Tokens: {xlm_roberta_tokens}")

# Demonstrate calling the tokenizer directly for BERT
bert_encoded = bert_tokenizer(
    sample_sentence,
    add_special_tokens=True,
    max_length=64,
    padding='max_length',
    truncation=True,
    return_attention_mask=True,
    return_tensors='pt' # Return PyTorch tensors
)

print("\nBERT Encoded (input_ids, attention_mask):")
print("Input IDs:", bert_encoded['input_ids'][0])
print("Attention Mask:", bert_encoded['attention_mask'][0])
print("Decoded (BERT):", bert_tokenizer.decode(bert_encoded['input_ids'][0]))

# Demonstrate calling the tokenizer directly for XLM-RoBERTa
xlm_roberta_encoded = xlm_roberta_tokenizer(
    sample_sentence,
    add_special_tokens=True,
    max_length=64,
    padding='max_length',
    truncation=True,
    return_attention_mask=True,
    return_tensors='pt' # Return PyTorch tensors
)

print("\nXLM-RoBERTa Encoded (input_ids, attention_mask):")
print("Input IDs:", xlm_roberta_encoded['input_ids'][0])
print("Attention Mask:", xlm_roberta_encoded['attention_mask'][0])
print("Decoded (XLM-RoBERTa):", xlm_roberta_tokenizer.decode(xlm_roberta_encoded['input_ids'][0]))

BERT Tokenizer loaded: bert-base-uncased
XLM-RoBERTa Tokenizer loaded: xlm-roberta-base

Sample Sentence: Hello, how are you today? This is an example sentence.

BERT Tokens: ['hello', ',', 'how', 'are', 'you', 'today', '?', 'this', 'is', 'an', 'example', 'sentence', '.']
XLM-RoBERTa Tokens: ['▁Hello', ',', '▁how', '▁are', '▁you', '▁today', '?', '▁This', '▁is', '▁an', '▁example', '▁sentence', '.']

BERT Encoded (input_ids, attention_mask):
Input IDs: tensor([ 101, 7592, 1010, 2129, 2024, 2017, 2651, 1029, 2023, 2003, 2019, 2742,
        6251, 1012,  102,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0])
Attention Mask: tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0,

### 3. Preparing Input Data for the Model - Special Tokens and Vocabulary
Understanding the special tokens and the vocabulary size of our tokenizers is key to correctly formatting input data for transformer models.

In [17]:
# Display BERT special tokens
print("\nBERT Special Tokens Map:")
print(bert_tokenizer.special_tokens_map)

# Display XLM-RoBERTa special tokens
print("\nXLM-RoBERta Special Tokens Map:")
print(xlm_roberta_tokenizer.special_tokens_map)

# Display BERT vocabulary size
print("\nBERT Vocabulary Size:", bert_tokenizer.vocab_size)

# Display XLM-RoBERTa vocabulary size
print("XLM-RoBERTa Vocabulary Size:", xlm_roberta_tokenizer.vocab_size)


BERT Special Tokens Map:
{'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}

XLM-RoBERta Special Tokens Map:
{'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'sep_token': '</s>', 'pad_token': '<pad>', 'cls_token': '<s>', 'mask_token': '<mask>'}

BERT Vocabulary Size: 30522
XLM-RoBERTa Vocabulary Size: 250002


### 3. Preparing Input Data for the Model - Tokenization of Dataset
Now, we'll apply the tokenization process to our actual training and testing data. We'll define a function to tokenize the 'premise' and 'hypothesis' columns for both BERT and XLM-RoBERTa, generating input IDs, attention masks, and token type IDs. This will prepare our data for model training.

In [18]:
import torch

MAX_LEN = 128 # Define a maximum sequence length

def tokenize_data(tokenizer, df, max_len=MAX_LEN):
    input_ids = []
    attention_masks = []
    token_type_ids = [] # BERT specific, will be all zeros for XLM-RoBERTa if not used

    for _, row in df.iterrows():
        # Use the tokenizer directly as a callable object
        encoded_dict = tokenizer(
            row['premise'],
            row['hypothesis'],
            add_special_tokens=True,
            max_length=max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_token_type_ids=True, # Required for BERT, for XLM-RoBERTa it will return a tensor of zeros
            return_tensors='pt' # Return PyTorch tensors
        )
        input_ids.append(encoded_dict['input_ids'])
        attention_masks.append(encoded_dict['attention_mask'])
        token_type_ids.append(encoded_dict['token_type_ids'])

    return torch.cat(input_ids, dim=0), torch.cat(attention_masks, dim=0), torch.cat(token_type_ids, dim=0)

print(f"Tokenizing training data with BERT tokenizer (max_len={MAX_LEN})...")
bert_train_input_ids, bert_train_attention_masks, bert_train_token_type_ids = tokenize_data(bert_tokenizer, train_df)
print("Done.")

print(f"Tokenizing test data with BERT tokenizer (max_len={MAX_LEN})...")
bert_test_input_ids, bert_test_attention_masks, bert_test_token_type_ids = tokenize_data(bert_tokenizer, test_df)
print("Done.")

print(f"\nTokenizing training data with XLM-RoBERTa tokenizer (max_len={MAX_LEN})...")
xlm_train_input_ids, xlm_train_attention_masks, xlm_train_token_type_ids = tokenize_data(xlm_roberta_tokenizer, train_df)
print("Done.")

print(f"Tokenizing test data with XLM-RoBERTa tokenizer (max_len={MAX_LEN})...")
xlm_test_input_ids, xlm_test_attention_masks, xlm_test_token_type_ids = tokenize_data(xlm_roberta_tokenizer, test_df)
print("Done.")

print("\nShape of BERT training input IDs:", bert_train_input_ids.shape)
print("Shape of XLM-RoBERTa training input IDs:", xlm_train_input_ids.shape)

Tokenizing training data with BERT tokenizer (max_len=128)...
Done.
Tokenizing test data with BERT tokenizer (max_len=128)...
Done.

Tokenizing training data with XLM-RoBERTa tokenizer (max_len=128)...
Done.
Tokenizing test data with XLM-RoBERTa tokenizer (max_len=128)...
Done.

Shape of BERT training input IDs: torch.Size([12120, 128])
Shape of XLM-RoBERTa training input IDs: torch.Size([12120, 128])


### 4. Creating PyTorch Dataset and DataLoader
To efficiently feed our tokenized data to the transformer models during training, we'll create custom PyTorch `Dataset` and `DataLoader` classes. This will allow for easy batching, shuffling, and loading of data.

In [25]:
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler

class CustomDataset(TensorDataset):
    def __init__(self, input_ids, attention_masks, token_type_ids, labels=None):
        self.input_ids = input_ids
        self.attention_masks = attention_masks
        self.token_type_ids = token_type_ids
        self.labels = labels

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        item = {
            'input_ids': self.input_ids[idx],
            'attention_mask': self.attention_masks[idx],
            'token_type_ids': self.token_type_ids[idx]
        }
        if self.labels is not None:
            # Corrected: Use .clone().detach() to avoid the UserWarning
            item['labels'] = self.labels[idx].clone().detach().to(torch.long)
        return item

# Convert labels to PyTorch tensor
train_labels = torch.tensor(train_df['label'].values, dtype=torch.long)

# Create BERT training and validation datasets using the first fold
# We'll use the first fold's indices for a single train/val split for this example
bert_train_fold_input_ids = bert_train_input_ids[train_indices[0]]
bert_train_fold_attention_masks = bert_train_attention_masks[train_indices[0]]
bert_train_fold_token_type_ids = bert_train_token_type_ids[train_indices[0]]
bert_train_fold_labels = train_labels[train_indices[0]]

bert_val_fold_input_ids = bert_train_input_ids[val_indices[0]]
bert_val_fold_attention_masks = bert_train_attention_masks[val_indices[0]]
bert_val_fold_token_type_ids = bert_train_token_type_ids[val_indices[0]]
bert_val_fold_labels = train_labels[val_indices[0]]

bert_train_dataset = CustomDataset(bert_train_fold_input_ids, bert_train_fold_attention_masks, bert_train_fold_token_type_ids, bert_train_fold_labels)
bert_val_dataset = CustomDataset(bert_val_fold_input_ids, bert_val_fold_attention_masks, bert_val_fold_token_type_ids, bert_val_fold_labels)

# The bert_test_dataset should remain without labels, as it's for final predictions
bert_test_dataset = CustomDataset(bert_test_input_ids, bert_test_attention_masks, bert_test_token_type_ids)

# Create XLM-RoBERTa datasets (keeping them as before for now, if not used in current error)
xlm_train_dataset = CustomDataset(xlm_train_input_ids, xlm_train_attention_masks, xlm_train_token_type_ids, train_labels)
xlm_test_dataset = CustomDataset(xlm_test_input_ids, xlm_test_attention_masks, xlm_test_token_type_ids)

BATCH_SIZE = 32

# Create BERT DataLoaders
bert_train_dataloader = DataLoader(
    bert_train_dataset,
    sampler=RandomSampler(bert_train_dataset),
    batch_size=BATCH_SIZE
)

bert_val_dataloader = DataLoader(
    bert_val_dataset,
    sampler=SequentialSampler(bert_val_dataset),
    batch_size=BATCH_SIZE
)

bert_test_dataloader = DataLoader(
    bert_test_dataset,
    sampler=SequentialSampler(bert_test_dataset),
    batch_size=BATCH_SIZE
)

# Create XLM-RoBERTa DataLoaders
xlm_train_dataloader = DataLoader(
    xlm_train_dataset,
    sampler=RandomSampler(xlm_train_dataset),
    batch_size=BATCH_SIZE
)

xlm_test_dataloader = DataLoader(
    xlm_test_dataset,
    sampler=SequentialSampler(xlm_test_dataset),
    batch_size=BATCH_SIZE
)

print(f"Created DataLoaders with batch size: {BATCH_SIZE}")
print(f"Number of BERT training batches: {len(bert_train_dataloader)}")
print(f"Number of BERT validation batches: {len(bert_val_dataloader)}")
print(f"Number of XLM-RoBERTa training batches: {len(xlm_train_dataloader)}")

Created DataLoaders with batch size: 32
Number of BERT training batches: 303
Number of BERT validation batches: 76
Number of XLM-RoBERTa training batches: 379


### 5. Fine-tuning BERT for Classification

Now, we'll load a pre-trained BERT model for sequence classification and prepare it for fine-tuning. This involves setting up the model, optimizer, and learning rate scheduler.

In [20]:
from transformers import BertForSequenceClassification, get_linear_schedule_with_warmup
import torch
import numpy as np
from torch.optim import AdamW # Corrected import for AdamW

# Define the number of labels (e.g., for 3-class classification)
num_labels = train_df['label'].nunique()

# Load BertForSequenceClassification, the pretrained BERT model with a classification layer on top.
bert_model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased", # Use the 12-layer BERT model, with an uncased vocab.
    num_labels = num_labels, # The number of output labels--2 for binary classification.
    output_attentions = False, # Whether the model returns attentions weights.
    output_hidden_states = False, # Whether the model returns all hidden-states.
)

# Tell pytorch to run this model on the GPU if available.
if torch.cuda.is_available():
    device = torch.device("cuda")
    bert_model.cuda()
    print("Using GPU for BERT training.")
else:
    device = torch.device("cpu")
    print("Using CPU for BERT training.")

# Get all of the model's parameters as a list of tuples.
params = list(bert_model.named_parameters())

print('The BERT model has {:} different named parameters.\n'.format(len(params)))

print('==== Embedding Layer ====\n')
for p in params[0:5]:
    print("{:<55} {:>12}".format(p[0], str(tuple(p[1].size()))))

print('\n==== First Transformer ====\n')
for p in params[5:10]:
    print("{:<55} {:>12}".format(p[0], str(tuple(p[1].size()))))

print('\n==== Output Layer ====\n')
for p in params[-4:]:
    print("{:<55} {:>12}".format(p[0], str(tuple(p[1].size()))))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Using GPU for BERT training.
The BERT model has 201 different named parameters.

==== Embedding Layer ====

bert.embeddings.word_embeddings.weight                  (30522, 768)
bert.embeddings.position_embeddings.weight                (512, 768)
bert.embeddings.token_type_embeddings.weight                (2, 768)
bert.embeddings.LayerNorm.weight                              (768,)
bert.embeddings.LayerNorm.bias                                (768,)

==== First Transformer ====

bert.encoder.layer.0.attention.self.query.weight          (768, 768)
bert.encoder.layer.0.attention.self.query.bias                (768,)
bert.encoder.layer.0.attention.self.key.weight            (768, 768)
bert.encoder.layer.0.attention.self.key.bias                  (768,)
bert.encoder.layer.0.attention.self.value.weight          (768, 768)

==== Output Layer ====

bert.pooler.dense.weight                                  (768, 768)
bert.pooler.dense.bias                                        (768,)
classifie

Next, we'll set up the optimizer and learning rate scheduler. We'll use `AdamW` optimizer and a linear scheduler with warm-up steps, which are commonly used for fine-tuning transformer models.

In [21]:
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW # Corrected import for AdamW

# Optimizer and learning rate scheduler
# AdamW is a class from the huggingface library (as opposed to pytorch's Adam).
optimizer = AdamW(bert_model.parameters(),
                  lr = 2e-5, # learning rate
                  eps = 1e-8 # Adam epsilon
                 )

# Number of training epochs.
epochs = 3

# Total number of training steps is [number of batches] x [number of epochs].
total_steps = len(bert_train_dataloader) * epochs

# Create the learning rate scheduler.
scheduler = get_linear_schedule_with_warmup(optimizer,
                                            num_warmup_steps = 0, # Default value in run_glue.py
                                            num_training_steps = total_steps)

print(f"BERT optimizer and scheduler configured for {epochs} epochs.")

BERT optimizer and scheduler configured for 3 epochs.


### 6. Training the BERT Model

Now, we'll define our training loop. This will involve iterating through our `DataLoader`, performing forward and backward passes, updating weights, and tracking metrics. We'll also define an evaluation function to assess the model's performance on the validation set.

In [26]:
import random
import time
import datetime
from sklearn.metrics import accuracy_score, f1_score

# Function to calculate accuracy and F1-score
def flat_accuracy(preds, labels):
    pred_flat = np.argmax(preds, axis=1).flatten()
    labels_flat = labels.flatten()
    return accuracy_score(labels_flat, pred_flat), f1_score(labels_flat, pred_flat, average='weighted')

# Helper function for formatting time
def format_time(elapsed):
    '''
    Takes a time in seconds and returns a string hh:mm:ss
    '''
    elapsed_rounded = int(round((elapsed)))
    return str(datetime.timedelta(seconds=elapsed_rounded))

# Set the seed for reproducible results
s_seed = 42 # Renamed to avoid conflict with seed_val which is used locally
random.seed(s_seed)
np.random.seed(s_seed)
torch.manual_seed(s_seed)
torch.cuda.manual_seed_all(s_seed)

# Store the average loss after each epoch so we can plot them.
loss_values = []

# Training loop
for epoch_i in range(0, epochs):

    # ========================================    Training    ========================================

    print(f"======== Epoch {epoch_i + 1} / {epochs} ========")
    print('Training...')

    t0 = time.time()

    total_loss = 0

    bert_model.train() # Put the model into training mode.

    # For each batch of training data...
    for step, batch in enumerate(bert_train_dataloader): # Use bert_train_dataloader for training

        # Progress update every 40 batches.
        if step % 40 == 0 and not step == 0:
            elapsed = format_time(time.time() - t0)
            print(f'  Batch {step:>5,}  of  {len(bert_train_dataloader):>5,}.    Elapsed: {elapsed}.')

        b_input_ids = batch['input_ids'].to(device)
        b_input_mask = batch['attention_mask'].to(device)
        b_token_type_ids = batch['token_type_ids'].to(device)
        b_labels = batch['labels'].to(device)

        bert_model.zero_grad()

        outputs = bert_model(
            b_input_ids,
            token_type_ids=b_token_type_ids,
            attention_mask=b_input_mask,
            labels=b_labels
        )

        loss = outputs.loss
        total_loss += loss.item()

        loss.backward()

        torch.nn.utils.clip_grad_norm_(bert_model.parameters(), 1.0) # Clip the norm of the gradients to 1.0 to prevent "exploding gradients"

        optimizer.step()
        scheduler.step()

    avg_train_loss = total_loss / len(bert_train_dataloader)

    loss_values.append(avg_train_loss)

    print("  Average training loss: {0:.2f}".format(avg_train_loss))
    print("  Training epoch took: {:} ".format(format_time(time.time() - t0)))

    # ========================================    Validation    ========================================

    print("\nRunning Validation...")

    t0 = time.time()

    bert_model.eval() # Put the model in evaluation mode

    eval_accuracy = 0
    eval_f1 = 0
    nb_eval_steps = 0

    # Evaluate data for one epoch
    for step, batch in enumerate(bert_val_dataloader): # Use bert_val_dataloader for validation

        b_input_ids = batch['input_ids'].to(device)
        b_input_mask = batch['attention_mask'].to(device)
        b_token_type_ids = batch['token_type_ids'].to(device)
        b_labels = batch['labels'].to(device)

        with torch.no_grad(): # Don't compute gradients
            outputs = bert_model(
                b_input_ids,
                token_type_ids=b_token_type_ids,
                attention_mask=b_input_mask
            )

        logits = outputs.logits.detach().cpu().numpy() # Move logits to CPU
        label_ids = b_labels.cpu().numpy() # Move labels to CPU (now b_labels will not be None)

        tmp_eval_accuracy, tmp_eval_f1 = flat_accuracy(logits, label_ids)

        eval_accuracy += tmp_eval_accuracy
        eval_f1 += tmp_eval_f1
        nb_eval_steps += 1

    print("  Accuracy: {0:.2f}".format(eval_accuracy / nb_eval_steps))
    print("  F1-Score: {0:.2f}".format(eval_f1 / nb_eval_steps))
    print("  Validation took: {:} ".format(format_time(time.time() - t0)))

print("\nTraining complete!")

======== Epoch 1 / 3 ========
Training...
  Batch    40  of    303.    Elapsed: 0:00:30.
  Batch    80  of    303.    Elapsed: 0:01:00.
  Batch   120  of    303.    Elapsed: 0:01:27.
  Batch   160  of    303.    Elapsed: 0:01:57.
  Batch   200  of    303.    Elapsed: 0:02:26.
  Batch   240  of    303.    Elapsed: 0:02:55.
  Batch   280  of    303.    Elapsed: 0:03:23.
  Average training loss: 0.49
  Training epoch took: 0:03:40 

Running Validation...
  Accuracy: 0.65
  F1-Score: 0.64
  Validation took: 0:00:19 
======== Epoch 2 / 3 ========
Training...
  Batch    40  of    303.    Elapsed: 0:00:29.
  Batch    80  of    303.    Elapsed: 0:00:58.
  Batch   120  of    303.    Elapsed: 0:01:26.
  Batch   160  of    303.    Elapsed: 0:01:55.
  Batch   200  of    303.    Elapsed: 0:02:24.
  Batch   240  of    303.    Elapsed: 0:02:53.
  Batch   280  of    303.    Elapsed: 0:03:22.
  Average training loss: 0.50
  Training epoch took: 0:03:38 

Running Validation...
  Accuracy: 0.65
  F1-Scor

### 7. Make Predictions on Test Data

After training, we will use the fine-tuned BERT model to make predictions on the unseen `test_df` data.

In [27]:
# Put model in evaluation mode
bert_model.eval()

predictions = []

# Predict
for batch in bert_test_dataloader:
    # Add batch to GPU
    batch = {k: v.to(device) for k, v in batch.items()}

    with torch.no_grad():
        # Forward pass, calculate logit predictions
        outputs = bert_model(
            batch['input_ids'],
            token_type_ids=batch['token_type_ids'],
            attention_mask=batch['attention_mask']
        )

    logits = outputs.logits

    # Move logits and labels to CPU
    logits = logits.detach().cpu().numpy()
    predictions.append(logits)

print('Prediction complete.')


Prediction complete.
